In [1]:
import os
from PIL import Image, ImageOps, ImageDraw, ImageFont
from tqdm import tqdm
import pandas as pd

In [2]:
def load_images(paths):
    imgs = []
    for p in paths:
        #try:
        im = Image.open(p).convert("RGB")
        imgs.append(im)
        #except Exception as e:
         #   print(f"Skipping {p}: {e}")
    return imgs
    
def save_three_on_one_pdf(images, texts, out_pdf="figures.pdf"):
    PAGE_W, PAGE_H = 2480, 3508  # A4 at 300 DPI
    MARGIN = 120
    ROW_GAP = 80
    CAPTION_H = 80   # space reserved for text
    BG_COLOR = "white"

    ims = [im.convert("RGB") if isinstance(im, Image.Image) else Image.open(im).convert("RGB") for im in images]
    if len(ims) != 3 or len(texts) != 3:
        raise ValueError("Expected exactly 3 images and 3 texts")

    usable_h = PAGE_H - 2*MARGIN - 2*ROW_GAP
    slot_h = usable_h // 3
    slot_w = PAGE_W - 2*MARGIN

    page = Image.new("RGB", (PAGE_W, PAGE_H), BG_COLOR)
    draw = ImageDraw.Draw(page)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Supplemental/Helvetica.ttc", 50)
    except:
        font = ImageFont.load_default()

    for idx, (im, txt) in enumerate(zip(ims, texts)):
        x0 = MARGIN
        y0 = MARGIN + idx * (slot_h + ROW_GAP)

        # --- FIX: use textbbox instead of textsize ---
        bbox = draw.textbbox((0, 0), txt, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        tx = x0 + (slot_w - tw) // 2
        draw.text((tx, y0), txt, fill="black", font=font)

        # Adjust slot so image sits below caption
        y0 += CAPTION_H
        w, h = im.size
        scale = min(slot_w / w, (slot_h - CAPTION_H) / h)
        im_fit = im.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        im_fit = ImageOps.expand(im_fit, border=2, fill="black")
        x = x0 + (slot_w - im_fit.width) // 2
        y = y0 + (slot_h - CAPTION_H - im_fit.height) // 2
        page.paste(im_fit, (x, y))

    page.save(out_pdf)
    # print(f"Saved {out_pdf}")

In [3]:
save_folder = 'Figs_combined'

In [4]:
plate_set = ['G-1040', 'G-1041', 'G-1042', 'GG-251']

for plate in plate_set:
    path_list = [
        # f'../data_GCDH_test_new/G2-30-58/{plate}/',
        f'../data_GCDH_test_new/G2-38-58/{plate}/',
        f'../data_GCDH_test_new/G3-30-58_thr/{plate}/',
        f'../data_GCDH_test_new/G_combined/{plate}/'
    ]

    path_to_save = f'{save_folder}/{plate}'
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)

    path = path_list[0] + 'fig_final/'
    well_list = os.listdir(path)
    well_list = [i for i in well_list if i.split('_')[1][0] == 'B']
    
    df_1 = pd.read_csv(os.path.join(path_list[0], 'result_table.csv'))
    df_2 = pd.read_csv(os.path.join(path_list[1], 'result_table.csv'))
    df_3 = pd.read_csv(os.path.join(path_list[2], 'result_table.csv'))
        
    for well in tqdm(well_list):
        path_list_ = [i + 'fig_final/' + well for i in path_list]
        images = load_images(path_list_)

        err_1 = df_1.loc[df_1['compound'] == well.split('_')[1], 'r_squared_1'].item()
        err_2 = df_2.loc[df_2['compound'] == well.split('_')[1], 'r_squared_1'].item()
        err_3 = df_3.loc[df_3['compound'] == well.split('_')[1], 'r_squared_1'].item()
        
        text = [f"G2-38-58: {plate} {well} err:{err_1}", f"G3-30-58_new: {well} err:{err_2}", f"G_combined: {plate} {well} err:{err_3}"]
        path = 'fig.pdf'        
        save_three_on_one_pdf(
            images, 
            text, 
            os.path.join(path_to_save, well.split('.')[0] + '.pdf')
        )

100%|███████████████████████████████████████████████████████████| 22/22 [00:01<00:00, 19.61it/s]
